In [1]:
import glob
import numpy as np
import json
from tqdm import tqdm
import torch
from pathlib import Path

In [17]:
JOINED_METADATA_PATH = "/home/wyf/orcd/pool/diffrhythm/metadata.json"

with open(JOINED_METADATA_PATH, "w") as fout:
    json.dump(clips, fout, indent=2)

print(f"{len(clips)=}")

print(clips[0]["activations_path"])
layer_act_shape = list(torch.load(clips[0]["activations_path"]).values())[0].shape[1:]
print(layer_act_shape)

len(clips)=399
/home/harinit9/orcd/pool/diffrhythm/activations/0000_chunk00.pt
torch.Size([6682, 2048])


In [18]:
# Downsample activations

def downsample(arr, target_length=256):
    arr = torch.tensor(arr)
    factor = arr.shape[0] // target_length
    arr = arr[:factor*target_length].reshape(target_length, factor, -1)
    arr = arr.max(dim=1).values
    return arr.numpy()

In [23]:
import glob
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

metadata_paths = glob.glob("/home/harinit9/orcd/pool/diffrhythm/metadata/*.json")
clips = []
for path in metadata_paths:
    with open(path) as fin:
        clip_batch = json.load(fin)
        for chunk in clip_batch["chunks"]:
            chunk["prompt"] = clip_batch["prompt"]
            chunk["full_audio_path"] = clip_batch["audio_path"]

    clips.extend(clip_batch["chunks"])

clips.sort(key=lambda x: x["chunk_id"])

JOINED_METADATA_PATH = "/home/wyf/orcd/pool/diffrhythm/metadata.json"
with open(JOINED_METADATA_PATH, "w") as fout:
    json.dump(clips, fout, indent=2)

print(f"Number of clips: {len(clips)}")

Number of clips: 399


In [24]:
layer_act_shape = (256, 2048)
ACTS_BY_LAYER_PATH = Path("/home/wyf/orcd/pool/diffrhythm/acts_by_layer")
ACTS_BY_LAYER_PATH.mkdir(exist_ok=True)

def downsample(arr, target_length=256):
    arr = arr.squeeze(0)
    seq_len = arr.shape[0]
    factor = max(seq_len // target_length, 1)
    trimmed = arr[:factor*target_length]
    reshaped = trimmed.reshape(target_length, factor, -1)
    return reshaped.max(dim=1).values.numpy()

layers = [12]
memmaps = {
    l: np.lib.format.open_memmap(
        ACTS_BY_LAYER_PATH / f"layer_{l:02d}.npy",
        mode="w+",
        dtype=np.float32,
        shape=(len(clips), *layer_act_shape)
        ) for l in layers
    }

for i, clip in enumerate(tqdm(clips)):
    acts = torch.load(clip["activations_path"])
    for l in layers:
        memmaps[l][i] = downsample(acts[f"dit.transformer_blocks.{l}"])
    del acts

100%|██████████| 399/399 [26:28<00:00,  3.98s/it]
